In [1]:
import os, torch, pickle
from transformers import AutoTokenizer
from src.models.ngram import NgramModel
from src.models.lstm import LSTMModel
from src.models.transformer import TransformerModel
from src.utils.config import CONFIG
from src.utils.utils import create_embedder, load_word2vec

/Users/JadenZ/miniconda3/envs/genz/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Sample code loading trained models

Edit src/utils/config.py or hyperparams as necessary

In [2]:
# Load saved models
out_dir = CONFIG["out_path"]
saved_model_dir = os.path.join(out_dir, "models")

# Ngram
n = 3
ngram_path = os.path.join(saved_model_dir, f"ngram_{n}.pt")
with open(ngram_path, "rb") as f:
    ngram_model: NgramModel = pickle.load(f)

# LSTM
epochs = 3
lr = 1e-5
lstm_path = os.path.join(saved_model_dir, f"lstm_{epochs}e_{lr}lr.pt")
state = torch.load(lstm_path)

embed_size = 256
src_w2v = load_word2vec(os.path.join(out_dir, f"src_embedding_{embed_size}.model"))
tgt_w2v = load_word2vec(os.path.join(out_dir, f"tgt_embedding_{embed_size}.model"))
src_embedder = create_embedder(src_w2v)
tgt_embedder = create_embedder(tgt_w2v)
lstm_model = LSTMModel(CONFIG, src_embedder, tgt_embedder)
lstm_model.load_state_dict(state)
lstm_model.eval()

# T5 Transformer
model_name = "t5-small"
epochs = 3
lr = 0.001
transformer_path = os.path.join(saved_model_dir, f"{model_name}_{epochs}e_{lr}lr.pt")
state = torch.load(transformer_path)

transformer_model = TransformerModel(CONFIG)
transformer_model.load_state_dict(state)
transformer_model.eval()


You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


TransformerModel(
  (model): T5ForConditionalGeneration(
    (shared): Embedding(32128, 512)
    (encoder): T5Stack(
      (embed_tokens): Embedding(32128, 512)
      (block): ModuleList(
        (0): T5Block(
          (layer): ModuleList(
            (0): T5LayerSelfAttention(
              (SelfAttention): T5Attention(
                (q): Linear(in_features=512, out_features=512, bias=False)
                (k): Linear(in_features=512, out_features=512, bias=False)
                (v): Linear(in_features=512, out_features=512, bias=False)
                (o): Linear(in_features=512, out_features=512, bias=False)
                (relative_attention_bias): Embedding(32, 8)
              )
              (layer_norm): T5LayerNorm()
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (1): T5LayerFF(
              (DenseReluDense): T5DenseActDense(
                (wi): Linear(in_features=512, out_features=2048, bias=False)
                (wo): Linear(in_fea

In [3]:
line = "to you all three,the senators alone of this great world,chief factors for the gods:i do not know wherefore my father should revengers want,having a son and friends,since julius caesar,who at philippi the good brutus ghosted,there saw you labouring for him."

ngram_translation = ngram_model.translate(line)
lstm_translation = lstm_model.translate(line)
transformer_translation = transformer_model.translate(line)

print("Original:     ", line)
print("Ngram:        ", ngram_translation)
print("LSTM:         ", lstm_translation)
print("Transformer : ", transformer_translation)

Original:      to you all three,the senators alone of this great world,chief factors for the gods:i do not know wherefore my father should revengers want,having a son and friends,since julius caesar,who at philippi the good brutus ghosted,there saw you labouring for him.
Ngram:         to you all three and the portia only of this big on and leave overcharged for the you a i do not know a my dad short god craving and and a son and if and since convo caesars and who you needed the good brutus while and win order you . for him you
LSTM:          bit used stanley sell ! knew what schooled if stream dog get hardass brain apemantus shake shook fairy humphrey sow adventures duke chit-chat keep friendships
Transformer :  to you all three, the senators alone of this great world, chief gods: i dont know where my father should go.
